In [ ]:
# Path config (auto-injected). Loads WORK_DIR from .env so notebooks run anywhere.
import os
from pathlib import Path
from dotenv import load_dotenv

_p = Path.cwd().resolve()
for _ in range(8):
    if (_p / '.env').is_file():
        load_dotenv(_p / '.env')
        break
    if _p == _p.parent:
        break
    _p = _p.parent

WORK_DIR = os.environ['WORK_DIR']
RETRIEVAL_DIR = f"{WORK_DIR}/retrieval_assets"

In [ ]:
# Cell 1: Setup
!pip install -q bert_score transformers sentence-transformers

import json
import numpy as np
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import torch

print(f"GPU Available: {torch.cuda.is_available()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Load Dataset
# Change path based on which dataset you're testing
DATASET = "squad"  # or "squad"

if DATASET == "hotpotqa":
    with open(f'{WORK_DIR}/llama 8b/hotpot/hotpot_qwen25_rag_length_controlled_1000_binmatched.json') as f:
        dataset = json.load(f)
else:
    with open(f'{WORK_DIR}/llama 8b/squad/hotpot_qwen25_rag_length_controlled_1000_binmatched.json') as f:
        dataset = json.load(f)

labels = np.array([s['hallucinated'] for s in dataset])
print(f"Loaded {len(dataset)} samples")
print(f"Hallucinated: {labels.sum()} / {len(labels)}")

In [ ]:
# Cell 3: BERTScore Baseline (OPTIMIZED - load model once)
from bert_score import BERTScorer

print("Computing BERTScore baseline...")
print("Loading BERTScore model (once)...")
scorer = BERTScorer(lang="en", rescale_with_baseline=True, device="cuda")

bertscore_scores = []

for sample in tqdm(dataset):
    context_str = " ".join(sample['context'])[:2000]
    P, R, F1 = scorer.score([sample['answer']], [context_str])
    bertscore_scores.append(1 - F1.item())  # Lower similarity = higher hallucination score

bertscore_auroc = roc_auc_score(labels, bertscore_scores)
print(f"\nBERTScore AUROC: {bertscore_auroc:.4f}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print("Loading NLI model (stable version)...")

model_name = "roberta-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to("cuda")
model.eval()

def compute_nli_score(context, answer):
    # STRONG truncation (important)
    context_str = " ".join(context)[:300]

    inputs = tokenizer(
        context_str,
        answer,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]

    # roberta-mnli label order: [contradiction, neutral, entailment]
    contradiction = probs[0].item()
    neutral = probs[1].item()

    # better scoring than your version
    return contradiction + 0.5 * neutral


print("Computing NLI baseline...")

nli_scores = []
valid_indices = []

for i, sample in enumerate(tqdm(dataset)):
    try:
        score = compute_nli_score(sample['context'], sample['answer'])
        nli_scores.append(score)
        valid_indices.append(i)
    except Exception:
        # DO NOT assign 0.5
        continue

nli_scores = np.array(nli_scores)
labels_matched = labels[valid_indices]

print(f"Valid samples: {len(nli_scores)} / {len(dataset)}")

nli_auroc = roc_auc_score(labels_matched, nli_scores)
print(f"\nNLI AUROC: {nli_auroc:.4f}")

In [ ]:
# Cell 5: Compute SDS Baseline
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Computing SDS baseline...")
sds_scores = []

for sample in tqdm(dataset):
    context_str = " ".join(sample['context'])
    answer = sample['answer']

    # Compute embeddings
    context_emb = embed_model.encode(context_str, convert_to_numpy=True)
    answer_emb = embed_model.encode(answer, convert_to_numpy=True)

    # SDS = L2 distance between context and answer embeddings
    sds = np.linalg.norm(context_emb - answer_emb)
    sds_scores.append(sds)

sds_auroc = roc_auc_score(labels, sds_scores)
print(f"\nSDS AUROC: {sds_auroc:.4f}")

In [ ]:
# Cell 7: Final Comparison

print("\n" + "="*50)
print(f"BASELINE COMPARISON ({DATASET.upper()})")
print("="*50)
print(f"\nMethod               AUROC")
print("-"*35)
print(f"NLI (RoBERTa-MNLI):  {nli_auroc:.4f}")
print(f"SDS (ours):          {sds_auroc:.4f}")
print(f"BERTScore:           {bertscore_auroc:.4f}")

print("-"*35)
print("\nNote: SDS + Uncertainty results in separate notebook (09-uncertainty-drift-kaggle.ipynb)")
print("="*50)

In [ ]:
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])  # Paste your token here


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
# Load Llama-3.1-8B-Instruct
model_name = "meta-llama/Llama-3.1-8B-Instruct"

print(f"Loading {model_name}...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Loaded {model_name}")
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print(f"✓ Loaded {model_name}")
print(f"✓ Device: {model.device}")


In [ ]:
# Cell: Compute Perplexity for all samples

print("\nComputing perplexity for all samples...")
print("Note: Reusing Phi-3 model from SelfCheckGPT section above")

def compute_perplexity_simple(question, context, answer):
    """Simplified perplexity computation"""
    prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"

    # Use chat template for Phi-3
    messages = [{"role": "user", "content": prompt}]
    prompt_formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = prompt_formatted + answer

    # Tokenize
    inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    prompt_inputs = tokenizer(prompt_formatted, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    prompt_len = prompt_inputs['input_ids'].shape[1]

    # Skip if answer is too short
    if inputs['input_ids'].shape[1] <= prompt_len:
        return None

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # Shape: (1, seq_len, vocab_size)

        # Get logits for answer tokens (shift by 1 for next-token prediction)
        answer_logits = logits[0, prompt_len-1:-1, :]  # (answer_len, vocab_size)
        answer_ids = inputs['input_ids'][0, prompt_len:]  # (answer_len,)

        # Skip if empty
        if len(answer_ids) == 0:
            return None

        # Compute probabilities
        probs = torch.softmax(answer_logits, dim=-1)
        token_probs = probs[range(len(answer_ids)), answer_ids]

        # Clamp to avoid numerical issues
        token_probs = torch.clamp(token_probs, min=1e-10)

        # Compute perplexity
        log_probs = torch.log(token_probs)
        perplexity = torch.exp(-log_probs.mean()).item()

        # Cap extreme values
        perplexity = min(perplexity, 100.0)

    return perplexity


perplexity_scores = []
valid_indices = []

for idx, sample in enumerate(tqdm(dataset)):
    context_str = " ".join(sample['context'])[:2000]

    try:
        ppl = compute_perplexity_simple(
            sample['question'],
            context_str,
            sample['answer']
        )

        if ppl is not None and not np.isnan(ppl) and not np.isinf(ppl):
            perplexity_scores.append(ppl)
            valid_indices.append(idx)
    except Exception as e:
        # Skip samples that cause errors
        continue

perplexity_scores = np.array(perplexity_scores)
valid_indices = np.array(valid_indices)

# Match labels to valid samples
labels_valid = labels[valid_indices]

print(f"\nProcessed {len(perplexity_scores)}/{len(dataset)} samples successfully")

ppl_auroc = roc_auc_score(labels_valid, perplexity_scores)
print(f"Perplexity AUROC: {ppl_auroc:.4f}")

# Combine SDS + Perplexity (on matched samples)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Match SDS scores to valid samples
sds_scores_valid = np.array(sds_scores)[valid_indices]

X = np.column_stack([sds_scores_valid, perplexity_scores])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=1000, C=0.1, random_state=42)
scores = cross_val_score(lr, X_scaled, labels_valid, cv=5, scoring='roc_auc')

combined_auroc = scores.mean()
combined_std = scores.std()

print(f"\nSDS + Perplexity: {combined_auroc:.4f} ± {combined_std:.4f}")

# Update final comparison
print("\n" + "="*60)
print(f"FINAL COMPARISON (HOTPOTQA) - n={len(labels_valid)} samples")
print("="*60)
print(f"Method               AUROC")
print("-"*35)
print(f"BERTScore:           {bertscore_auroc:.4f}")
print(f"SDS:                 {sds_auroc:.4f}")
print(f"NLI:                 {nli_auroc:.4f}")
print(f"LettuceDetect:       0.7231")
print(f"Perplexity:          {ppl_auroc:.4f}")
print(f"SDS + Perplexity:    {combined_auroc:.4f} ± {combined_std:.4f}")
print("="*60)

if combined_auroc > 0.7231:
    improvement = combined_auroc - 0.7231
    pct = (improvement / 0.7231) * 100
    print(f"\nYour method beats LettuceDetect by +{improvement:.4f} ({pct:.1f}%)")


In [ ]:
# Add this diagnostic cell BEFORE the combination:

print("\n=== DIAGNOSTIC ===")
print(f"SDS scores range: [{sds_scores_valid.min():.3f}, {sds_scores_valid.max():.3f}]")
print(f"Perplexity range: [{perplexity_scores.min():.3f}, {perplexity_scores.max():.3f}]")
print(f"SDS mean by class:")
print(f"  Non-hall: {sds_scores_valid[labels_valid==0].mean():.3f}")
print(f"  Hall:     {sds_scores_valid[labels_valid==1].mean():.3f}")
print(f"Perplexity mean by class:")
print(f"  Non-hall: {perplexity_scores[labels_valid==0].mean():.3f}")
print(f"  Hall:     {perplexity_scores[labels_valid==1].mean():.3f}")

# Check correlation
from scipy.stats import pearsonr
corr, p = pearsonr(sds_scores_valid, perplexity_scores)
print(f"\nCorrelation (SDS vs Perplexity): r={corr:.3f}, p={p:.6f}")


In [ ]:
# Instead of logistic regression, just try a weighted average
print("\n=== SIMPLE COMBINATION (WEIGHTED AVERAGE) ===")

# Normalize both to 0-1 range
sds_norm = (sds_scores_valid - sds_scores_valid.min()) / (sds_scores_valid.max() - sds_scores_valid.min())
ppl_norm = (perplexity_scores - perplexity_scores.min()) / (perplexity_scores.max() - perplexity_scores.min())

# Try different weights
for w in [0.3, 0.5, 0.7]:
    combined = w * sds_norm + (1-w) * ppl_norm
    auroc = roc_auc_score(labels_valid, combined)
    print(f"Weight SDS={w:.1f}, PPL={1-w:.1f}: AUROC={auroc:.4f}")

# Best simple combination
combined_simple = 0.3 * sds_norm + 0.7 * ppl_norm
simple_auroc = roc_auc_score(labels_valid, combined_simple)
print(f"\nBest simple combination: {simple_auroc:.4f}")


In [ ]:
# Cell: FIXED Combination with Better Scaling

print("\n=== IMPROVED COMBINATION ===")

# Log-transform perplexity to compress the range
perplexity_log = np.log(perplexity_scores)

print(f"Log-perplexity range: [{perplexity_log.min():.3f}, {perplexity_log.max():.3f}]")

# Now combine with logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

X = np.column_stack([sds_scores_valid, perplexity_log])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)  # Changed C=1.0
scores = cross_val_score(lr, X_scaled, labels_valid, cv=5, scoring='roc_auc')

combined_auroc = scores.mean()
combined_std = scores.std()

print(f"SDS + Log(Perplexity): {combined_auroc:.4f} ± {combined_std:.4f}")

# Also try just using perplexity features alone with different transformations
print("\n=== PERPLEXITY-ONLY VARIANTS ===")

# Original perplexity
ppl_auroc = roc_auc_score(labels_valid, perplexity_scores)
print(f"Perplexity (raw):        {ppl_auroc:.4f}")

# Log perplexity
ppl_log_auroc = roc_auc_score(labels_valid, perplexity_log)
print(f"Perplexity (log):        {ppl_log_auroc:.4f}")

# Inverse perplexity (like probability)
ppl_inv = 1.0 / perplexity_scores
ppl_inv_auroc = roc_auc_score(labels_valid, -ppl_inv)  # Negate because lower PPL = better
print(f"Perplexity (1/x):        {ppl_inv_auroc:.4f}")

# Updated final comparison
print("\n" + "="*60)
print(f"FINAL COMPARISON (HOTPOTQA) - n={len(labels_valid)} samples")
print("="*60)
print(f"Method                    AUROC")
print("-"*40)
print(f"BERTScore:                {bertscore_auroc:.4f}")
print(f"SDS:                      {sds_auroc:.4f}")
print(f"NLI:                      {nli_auroc:.4f}")
print(f"LettuceDetect:            0.7231")
print(f"Perplexity:               {ppl_auroc:.4f}")
print(f"SDS + Perplexity:         {combined_auroc:.4f}")
print("="*60)


In [ ]:
# Cell: Save results for LettuceDetect comparison

# Fit the model on all data (for saving predictions)
lr_final = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_final.fit(X_scaled, labels_valid)
combined_scores = lr_final.predict_proba(X_scaled)[:, 1]

results = {
    'method': 'SDS + Log(Perplexity)',
    'auroc': combined_auroc,
    'auroc_std': combined_std,
    'dataset': 'hotpotqa_binmatched',
    'n_samples': len(labels_valid),
    'labels': labels_valid,
    'sds_scores': sds_scores_valid,
    'perplexity_scores': perplexity_scores,
    'combined_scores': combined_scores,  # Now from fitted model
    'baselines': {
        'bertscore': bertscore_auroc,
        'sds': sds_auroc,
        'nli': nli_auroc,
        'lettucedetect': 0.7231,
        'perplexity': ppl_auroc
    }
}

import pickle
with open('hotpot_sds_perplexity_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print("\n✓ Saved results to hotpot_sds_perplexity_results.pkl")
print(f"  - Method: {results['method']}")
print(f"  - AUROC: {results['auroc']:.4f} ± {results['auroc_std']:.4f}")
print(f"  - Samples: {results['n_samples']}")
